# 1. input & input embedding

In [2]:
import math
import torch
import torch.nn as nn

# 符号约定：B = batch 大小，n = 句子长度（token 数），d = d_model，V = 词表大小


class TokenEmbedding(nn.Module):
    """
    输入:  ids  [B, n]     整数 token ID，取值范围是 0 ~ V-1
    输出:  x    [B, n, d]  每个 token 对应的向量，并乘以 √d
    参数:  一张 V×d 的可学习表
    """
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        # 1.1: 创建一张 vocab_size × d_model 的 Embedding 表
        vocab = {"i": 1, 'like':2, 'reading':3, 'playing':4}
        self.emb = nn.Embedding(vocab_size, d_model)

    def forward(self, ids):
        # TODO 1.2: 用 ids 查表，得到 [B, n, d]，再乘以 √d_model
        #   提示：self.emb(ids) 就是查表；开根号用 math.sqrt
        return self.emb(ids) * math.sqrt(self.d_model)
        


# ============ 自测：不要修改 ============
if __name__ == "__main__":
    torch.manual_seed(0)
    vocab = {"<pad>": 0, "我": 1, "爱": 2, "学习": 3, "吃饭": 4}
    ids = torch.tensor([[1, 2, 3],      # 我 爱 学习
                        [1, 4, 0]])     # 我 吃饭 <pad>
    V, d = len(vocab), 4

    def check(name, cond):
        print(("✅ " if cond else "❌ ") + name)

    emb = TokenEmbedding(V, d)
    out = emb(ids)

    check("表的形状是 [V, d] = [5, 4]", emb.emb.weight.shape == (V, d))
    check("输出形状是 [B, n, d] = [2, 3, 4]", out.shape == (2, 3, d))
    check("「我」= 表的第 1 行 × √d", torch.allclose(out[0, 0], emb.emb.weight[1] * math.sqrt(d)))
    check("两句话里的「我」向量相同", torch.allclose(out[0, 0], out[1, 0]))
    check("表是可训练参数", emb.emb.weight.requires_grad)
    print("参数量:", sum(p.numel() for p in emb.parameters()), "（应为 V×d = 20）")

✅ 表的形状是 [V, d] = [5, 4]
✅ 输出形状是 [B, n, d] = [2, 3, 4]
✅ 「我」= 表的第 1 行 × √d
✅ 两句话里的「我」向量相同
✅ 表是可训练参数
参数量: 20 （应为 V×d = 20）


# 2. Positional Encoding

In [ ]:
import math
import torch
import torch.nn as nn

# 符号约定：B = batch 大小，n = 句子长度，d = d_model


class PositionalEncoding(nn.Module):
    """
    输入:  x  [B, n, d]   Embedding 之后的词向量
    输出:  x  [B, n, d]   词向量 + 位置向量（形状不变）

    公式:  PE(pos, 2i)   = sin(pos / 10000^(2i/d))    偶数维用 sin
           PE(pos, 2i+1) = cos(pos / 10000^(2i/d))    奇数维用 cos

    思路:  在 __init__ 里一次性算好一张 [max_len, d] 的位置编码表，
           forward 时只取前 n 行，加到 x 上。
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)                    # [max_len, d]，先全填 0
        pos = torch.arange(max_len).unsqueeze(1).float()      # [max_len, 1]，即 0,1,2,...,max_len-1 竖着排

        # TODO 2.1: 算出公式里的「2i」，也就是所有偶数维的下标 0, 2, 4, ..., d-2
        #   提示：torch.arange(起点, 终点, 步长)
        #   形状应为 [d/2]
        two_i = torch.arange(0, d_model, 2)

        # TODO 2.2: 算出分母 10000^(2i/d)
        #   提示：用上面的 two_i 和 d_model；幂运算用 **
        #   形状应为 [d/2]
        div = 10000 ** (two_i / d_model)

        # TODO 2.3: 偶数列填 sin(pos / div)，奇数列填 cos(pos / div)
        #   提示：pe[:, 0::2] 表示「所有行、从第 0 列开始每隔一列取」，也就是偶数列
        #         pe[:, 1::2] 就是奇数列
        #   提示：pos 是 [max_len, 1]，div 是 [d/2]，两者相除会自动广播成 [max_len, d/2]
        angle = pos / div    # [max_len, 1] ÷ [d/2] → 广播成 [max_len, d/2]
        pe[:, 0::2] = torch.sin(angle)
        pe[:, 1::2] = torch.cos(angle)


        # 不用改：把 pe 存进模型。它是固定值、不参与训练，所以用 buffer 而不是 Parameter
        self.register_buffer("pe", pe)

    def forward(self, x):
        # TODO 2.4: 取 pe 的前 n 行（n = x 的第 1 维大小），加到 x 上并返回
        #   提示：x.size(1) 就是 n；pe[:n] 的形状是 [n, d]，和 [B, n, d] 相加会自动广播
        n = x.size(1)
        return x + self.pe[:n]


# ============ 自测：不要修改 ============
if __name__ == "__main__":
    d, n, B = 8, 5, 2
    pe_layer = PositionalEncoding(d, max_len=100)
    x = torch.zeros(B, n, d)                 # 用全 0 输入，输出就是纯位置编码，方便检查
    out = pe_layer(x)

    def check(name, cond):
        print(("✅ " if cond else "❌ ") + name)

    check("pe 表形状是 [max_len, d] = [100, 8]", pe_layer.pe.shape == (100, d))
    check("输出形状不变 [B, n, d] = [2, 5, 8]", out.shape == (B, n, d))
    check("pos=0：偶数维全是 0（sin0）", torch.allclose(out[0, 0, 0::2], torch.zeros(d // 2)))
    check("pos=0：奇数维全是 1（cos0）", torch.allclose(out[0, 0, 1::2], torch.ones(d // 2)))
    check("pos=1，第 0 维 = sin(1)", abs(out[0, 1, 0].item() - math.sin(1)) < 1e-5)
    check("pos=1，第 1 维 = cos(1)", abs(out[0, 1, 1].item() - math.cos(1)) < 1e-5)
    check("pos=3，第 2 维 = sin(3 / 10000^(2/8))", abs(out[0, 3, 2].item() - math.sin(3 / 10000 ** (2 / 8))) < 1e-5)
    check("两句话加的位置编码相同", torch.allclose(out[0], out[1]))
    check("pe 不是可训练参数", len(list(pe_layer.parameters())) == 0)

    x2 = torch.ones(B, n, d)
    check("是「相加」而不是替换", torch.allclose(pe_layer(x2), out + 1))

✅ pe 表形状是 [max_len, d] = [100, 8]
✅ 输出形状不变 [B, n, d] = [2, 5, 8]
✅ pos=0：偶数维全是 0（sin0）
✅ pos=0：奇数维全是 1（cos0）
✅ pos=1，第 0 维 = sin(1)
✅ pos=1，第 1 维 = cos(1)
✅ pos=3，第 2 维 = sin(3 / 10000^(2/8))
✅ 两句话加的位置编码相同
✅ pe 不是可训练参数
✅ 是「相加」而不是替换


# 3. single-head attention

In [3]:
import math, torch, torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

    def forward(self, x):            # x: [B, n, d]
        d = x.size(-1)
        # TODO 3.1: 用 W_q、W_k、W_v 算出 Q、K、V，形状都是 [B, n, d]
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        # TODO 3.2: 算分数 Q·Kᵀ / √d，形状 [B, n, n]
        score = Q @ K.transpose(-2, -1) / math.sqrt(d)
        # TODO 3.3: 对最后一维做 softmax，得到权重
        weight = torch.softmax(score, dim = -1)
        # TODO 3.4: 权重 @ V，返回 [B, n, d]
        return weight @ V

x = torch.randn(2, 5, 8)
out = SelfAttention(8)(x)
print(out.shape)   # 应该是 torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


# 4. Multi-Head Attention

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):                 # x: [B, n, d]
        B, n, d = x.shape
        # TODO 4.1: 算 Q、K、V，并切成 [B, h, n, d_k]
        #   提示：self.W_q(x).view(B, n, self.h, self.d_k).transpose(1, 2)
        Q = self.W_q(x).view(B, n, self.h, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, n, self.h, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, n, self.h, self.d_k).transpose(1, 2)
        # TODO 4.2: 分数 Q·Kᵀ / √d_k，形状 [B, h, n, n]
        score = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        # TODO 4.3: softmax 得到权重，再 @ V，形状 [B, h, n, d_k]
        weight = torch.softmax(score, dim = -1)
        attention = weight @ V
        # TODO 4.4: 拼回 [B, n, d]
        #   提示：.transpose(1, 2).reshape(B, n, d)
        attention = attention.transpose(1, 2).reshape(B, n, d)
        # TODO 4.5: 过 W_o 并返回
        return self.W_o(attention)

x = torch.randn(2, 5, 8)
out = MultiHeadAttention(8, h=2)(x)
print(out.shape)   # 应该是 torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


# 5. FFN

In [6]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # TODO 5.1: 建两个 Linear 层：d_model → d_ff，d_ff → d_model
        #   提示：像之前 W_q 那样，存成 self.linear1、self.linear2
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        # TODO 5.2: 建一个 ReLU 层
        #   提示：nn.ReLU()，不用传参数
        self.relu = nn.ReLU()

    def forward(self, x):          # x: [B, n, d]
        # TODO 5.3: 依次过 linear1 → relu → linear2，返回结果
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

x = torch.randn(2, 5, 8)
out = FeedForward(8, d_ff=32)(x)
print(out.shape)   # 应该是 torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


# 6. Add & Norm

In [8]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        # TODO 6.1: 造一个多头注意力层 self.attn 和一个 FFN 层 self.ffn
        #   提示：MultiHeadAttention(d_model, h)、FeedForward(d_model, d_ff)
        self.attn = MultiHeadAttention(d_model, h)
        self.ffn = FeedForward(d_model, d_ff)

        # TODO 6.2: 造两个 LayerNorm：self.norm1、self.norm2
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):          # x: [B, n, d]
        # TODO 6.3: x = norm1( x + 注意力(x) )
        x = self.norm1(x + self.attn(x))
        # TODO 6.4: x = norm2( x + FFN(x) )，然后 return x
        x = self.norm2(x + self.ffn(x))

        return x

x = torch.randn(2, 5, 8)
out = TransformerBlock(8, h=2, d_ff=32)(x)
print(out.shape)   # 应该是 torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


# 7. Nx

In [ ]:
class Encoder(nn.Module):
    def __init__(self, d_model, h, d_ff, N):
        super().__init__()
        # 7.1: 造 N 个 TransformerBlock 存进 self.layers
        self.layers = nn.ModuleList([TransformerBlock(d_model, h, d_ff) for _ in range(N)])

    def forward(self, x):          # x: [B, n, d]
        # 7.2: 让 x 依次过每一层
        for layer in self.layers:
            x = layer(x)
        return x

x = torch.randn(2, 5, 8)
out = Encoder(8, h=2, d_ff=32, N=6)(x)
print(out.shape)   # torch.Size([2, 5, 8])